# 5 Beginner Quant Projects — Google Colab Notebook

A step-by-step, run-in-order notebook covering:

1. Stock Data Fetcher & Moving Average Visualizer
2. SMA Crossover Backtester
3. Black-Scholes Option Pricer
4. Markowitz Portfolio Optimization (Efficient Frontier)
5. Value at Risk (VaR) Calculator

**How to use:** Open this in Google Colab (`File > Upload notebook`), then run cells top to bottom with Shift+Enter. Each project builds understanding step by step — read the markdown before each code cell, don't just run blindly.


## Setup — run this first

Installs/imports everything used across all 5 projects.

In [ ]:
!pip install yfinance --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.style.use('seaborn-v0_8-darkgrid') if 'seaborn-v0_8-darkgrid' in plt.style.available else None
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')


---
# Project 1: Stock Data Fetcher & Moving Average Visualizer

**Goal:** learn to pull market data and compute basic technical indicators.

**Step 1.1 — Pull daily OHLCV data for a ticker.**


In [ ]:
TICKER = "AAPL"   # change this to any ticker you like
START = "2020-01-01"
END = None        # None = up to today

df = yf.download(TICKER, start=START, end=END, auto_adjust=True)
df = df[['Close']].dropna()
df.columns = ['Close']
df.head()


**Step 1.2 — Compute 20-day and 50-day Simple Moving Averages (SMA).**

In [ ]:
df['SMA20'] = df['Close'].rolling(20).mean()
df['SMA50'] = df['Close'].rolling(50).mean()
df.tail()


**Step 1.3 — Plot price with both SMAs.**

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df.index, df['Close'], label='Close', linewidth=1)
plt.plot(df.index, df['SMA20'], label='SMA 20', linewidth=1.2)
plt.plot(df.index, df['SMA50'], label='SMA 50', linewidth=1.2)
plt.title(f'{TICKER} Price with 20/50-day SMAs')
plt.xlabel('Date'); plt.ylabel('Price')
plt.legend()
plt.show()


**Step 1.4 — Build a signal column.**

Signal = 1 (bullish / "in the market") when SMA20 > SMA50, else 0.
This is the classic "golden cross / death cross" logic.


In [ ]:
df['Signal'] = np.where(df['SMA20'] > df['SMA50'], 1, 0)
df[['Close','SMA20','SMA50','Signal']].tail(10)


**Beginner checkpoint:** A crossover signal is a trend-following, lagging indicator — it confirms a trend has already started rather than predicting one. Faster SMA vs. slower SMA is a classic noise-vs-lag trade-off: shorter windows react faster but generate more false signals ("whipsaws"); longer windows are smoother but slower to react.

---
# Project 2: SMA Crossover Backtester

**Goal:** turn Project 1's signal into a strategy with a measurable P&L, and avoid the #1 beginner mistake — lookahead bias.

**Step 2.1 — Compute daily returns.**


In [ ]:
df['Return'] = df['Close'].pct_change()
df[['Close','Return']].tail()


**Step 2.2 — Compute strategy return.**

CRITICAL: use `Signal.shift(1)` — you can only trade on **yesterday's** signal, since today's close (which generates today's signal) isn't known until the market closes. Using `Signal` without `.shift(1)` is lookahead bias and will make your backtest look artificially good.


In [ ]:
df['Strategy_Return'] = df['Signal'].shift(1) * df['Return']
df[['Return','Signal','Strategy_Return']].tail()


**Step 2.3 — Compute cumulative returns and compare to buy-and-hold.**

In [ ]:
df['Cum_BuyHold'] = (1 + df['Return']).cumprod()
df['Cum_Strategy'] = (1 + df['Strategy_Return']).cumprod()

plt.figure(figsize=(12,6))
plt.plot(df.index, df['Cum_BuyHold'], label='Buy & Hold')
plt.plot(df.index, df['Cum_Strategy'], label='SMA Crossover Strategy')
plt.title(f'{TICKER}: Strategy vs Buy & Hold (cumulative growth of $1)')
plt.xlabel('Date'); plt.ylabel('Growth of $1')
plt.legend()
plt.show()


**Step 2.4 — Compute the Sharpe ratio (risk-adjusted return, annualized).**

In [ ]:
def sharpe_ratio(returns, periods_per_year=252, risk_free=0.0):
    excess = returns.dropna() - risk_free/periods_per_year
    return (excess.mean() / excess.std()) * np.sqrt(periods_per_year)

sharpe_strategy = sharpe_ratio(df['Strategy_Return'])
sharpe_buyhold  = sharpe_ratio(df['Return'])

print(f"Strategy Sharpe:  {sharpe_strategy:.3f}")
print(f"Buy&Hold Sharpe:  {sharpe_buyhold:.3f}")


**Beginner checkpoint:** A strategy that beats buy-and-hold on cumulative return but has a *lower* Sharpe ratio took more risk per unit of return — that's not necessarily a better strategy. Always compare risk-adjusted, not just raw return.

---
# Project 3: Black-Scholes Option Pricer

**Goal:** learn options fundamentals and closed-form pricing.

**Step 3.1 — Implement the Black-Scholes formula for European call & put.**

Inputs: spot price (S), strike (K), time to maturity in years (T), risk-free rate (r), volatility (σ).


In [ ]:
def black_scholes(S, K, T, r, sigma, option_type='call'):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)

    if option_type == 'call':
        price = S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)
    elif option_type == 'put':
        price = K*np.exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)
    else:
        raise ValueError("option_type must be 'call' or 'put'")
    return price, d1, d2

# Example: S=100, K=100 (at-the-money), 1 year to expiry, 5% risk-free rate, 20% vol
S, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.20
call_price, d1, d2 = black_scholes(S, K, T, r, sigma, 'call')
put_price, _, _ = black_scholes(S, K, T, r, sigma, 'put')

print(f"Call price: {call_price:.4f}")
print(f"Put price:  {put_price:.4f}")


**Step 3.2 — Validate: put-call parity.**

A useful sanity check that doesn't require an external calculator: `Call - Put = S - K*e^(-rT)` should hold exactly for European options.


In [ ]:
lhs = call_price - put_price
rhs = S - K*np.exp(-r*T)
print(f"Call - Put = {lhs:.6f}")
print(f"S - K*e^-rT = {rhs:.6f}")
print("Match:", np.isclose(lhs, rhs))


**Step 3.3 — Add the Greeks (Delta, Gamma, Vega, Theta).**

In [ ]:
def greeks(S, K, T, r, sigma, option_type='call'):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)

    gamma = norm.pdf(d1) / (S*sigma*np.sqrt(T))
    vega  = S*norm.pdf(d1)*np.sqrt(T) / 100   # per 1% vol change

    if option_type == 'call':
        delta = norm.cdf(d1)
        theta = (-(S*norm.pdf(d1)*sigma)/(2*np.sqrt(T)) - r*K*np.exp(-r*T)*norm.cdf(d2)) / 365
    else:
        delta = norm.cdf(d1) - 1
        theta = (-(S*norm.pdf(d1)*sigma)/(2*np.sqrt(T)) + r*K*np.exp(-r*T)*norm.cdf(-d2)) / 365

    return {'Delta': delta, 'Gamma': gamma, 'Vega': vega, 'Theta (per day)': theta}

print("Call Greeks:", greeks(S, K, T, r, sigma, 'call'))
print("Put Greeks: ", greeks(S, K, T, r, sigma, 'put'))


**Step 3.4 — Plot option price vs. underlying spot price (the payoff curve).**

In [ ]:
spots = np.linspace(50, 150, 100)
call_prices = [black_scholes(s, K, T, r, sigma, 'call')[0] for s in spots]
put_prices  = [black_scholes(s, K, T, r, sigma, 'put')[0] for s in spots]

plt.figure(figsize=(10,6))
plt.plot(spots, call_prices, label='Call Price')
plt.plot(spots, put_prices, label='Put Price')
plt.axvline(K, color='gray', linestyle='--', label='Strike (K)')
plt.title('Black-Scholes Option Price vs. Spot Price')
plt.xlabel('Spot Price (S)'); plt.ylabel('Option Price')
plt.legend()
plt.show()


**Beginner checkpoint:**
- **Delta** — sensitivity of option price to a $1 move in the underlying (also ~ probability of finishing in-the-money).
- **Gamma** — how fast Delta itself changes (convexity).
- **Vega** — sensitivity to a 1% change in implied volatility.
- **Theta** — value lost per day purely from time passing ("time decay"), all else equal.

---
# Project 4: Markowitz Portfolio Optimization (Efficient Frontier)

**Goal:** learn mean-variance optimization — how combining assets reduces risk.

**Step 4.1 — Pick a basket of stocks and pull historical prices.**


In [ ]:
TICKERS = ['AAPL', 'MSFT', 'JPM', 'XOM', 'JNJ']
START = '2020-01-01'

prices = yf.download(TICKERS, start=START, auto_adjust=True)['Close'].dropna()
prices.head()


**Step 4.2 — Compute daily returns, then annualize mean returns and covariance.**

In [ ]:
returns = prices.pct_change().dropna()

mean_returns_annual = returns.mean() * 252
cov_matrix_annual = returns.cov() * 252

print("Annualized mean returns:\n", mean_returns_annual)
print("\nAnnualized covariance matrix:\n", cov_matrix_annual)


**Step 4.3 — Simulate thousands of random portfolios.**

For each random weight combination (summing to 1): compute expected return and volatility.
- Portfolio return = `w @ mean_returns`
- Portfolio volatility = `sqrt(w.T @ cov_matrix @ w)`


In [ ]:
np.random.seed(42)
n_portfolios = 5000
n_assets = len(TICKERS)

results = np.zeros((3, n_portfolios))  # row0=return, row1=volatility, row2=sharpe
weights_record = []

for i in range(n_portfolios):
    w = np.random.random(n_assets)
    w /= np.sum(w)
    weights_record.append(w)

    port_return = np.dot(w, mean_returns_annual)
    port_vol = np.sqrt(np.dot(w.T, np.dot(cov_matrix_annual, w)))

    results[0, i] = port_return
    results[1, i] = port_vol
    results[2, i] = port_return / port_vol   # Sharpe assuming 0% risk-free rate


**Step 4.4 — Plot the efficient frontier and mark the best portfolios.**

In [ ]:
max_sharpe_idx = np.argmax(results[2])
min_vol_idx = np.argmin(results[1])

plt.figure(figsize=(12,7))
sc = plt.scatter(results[1], results[0], c=results[2], cmap='viridis', s=8, alpha=0.6)
plt.colorbar(sc, label='Sharpe Ratio')
plt.scatter(results[1, max_sharpe_idx], results[0, max_sharpe_idx],
            marker='*', color='red', s=400, label='Max Sharpe Portfolio')
plt.scatter(results[1, min_vol_idx], results[0, min_vol_idx],
            marker='*', color='blue', s=400, label='Min Volatility Portfolio')
plt.title('Efficient Frontier — Random Portfolios')
plt.xlabel('Annualized Volatility')
plt.ylabel('Annualized Return')
plt.legend()
plt.show()

print("Max Sharpe portfolio weights:")
for t, w in zip(TICKERS, weights_record[max_sharpe_idx]):
    print(f"  {t}: {w:.2%}")

print("\nMin Volatility portfolio weights:")
for t, w in zip(TICKERS, weights_record[min_vol_idx]):
    print(f"  {t}: {w:.2%}")


**Beginner checkpoint:** Diversification reduces portfolio variance even when every individual asset is risky, because portfolio variance depends on the **covariance** between assets, not just their individual variances. Assets that move somewhat independently (or inversely) smooth out the combined ride even if none of them is safe alone.

---
# Project 5: Value at Risk (VaR) Calculator

**Goal:** learn basic risk measurement using three different methods on the same portfolio.

We'll use the Max Sharpe portfolio weights from Project 4.

**Step 5.1 — Build the portfolio's historical daily return series.**


In [ ]:
port_weights = weights_record[max_sharpe_idx]
portfolio_returns = returns.dot(port_weights)
portfolio_returns.tail()


**Step 5.2 — Historical VaR.**

Sort actual historical returns and take the 5th percentile — no distribution assumption required.


In [ ]:
confidence = 0.95
historical_var = np.percentile(portfolio_returns, (1 - confidence) * 100)
print(f"1-day Historical VaR at {confidence:.0%} confidence: {historical_var:.4%}")
print(f"(i.e., on 5% of days, you'd expect to lose more than {abs(historical_var):.2%} of the portfolio)")


**Step 5.3 — Parametric (Variance-Covariance) VaR.**

Assumes returns are normally distributed: `VaR = mean - z*std`, with z = 1.65 for 95% confidence.


In [ ]:
mu = portfolio_returns.mean()
sigma_p = portfolio_returns.std()
z = norm.ppf(1 - confidence)   # ~ -1.645 for 95%

parametric_var = mu + z * sigma_p
print(f"1-day Parametric VaR at {confidence:.0%} confidence: {parametric_var:.4%}")


**Step 5.4 — Monte Carlo VaR.**

Simulate thousands of possible next-day returns using the historical mean/std, then take the 5th percentile of the simulated outcomes.

In [ ]:
np.random.seed(1)
n_sims = 100000
simulated_returns = np.random.normal(mu, sigma_p, n_sims)
mc_var = np.percentile(simulated_returns, (1 - confidence) * 100)

print(f"1-day Monte Carlo VaR at {confidence:.0%} confidence: {mc_var:.4%}")

print("\n--- Comparison ---")
print(f"Historical VaR:  {historical_var:.4%}")
print(f"Parametric VaR:  {parametric_var:.4%}")
print(f"Monte Carlo VaR: {mc_var:.4%}")


**Step 5.5 — Extend to Conditional VaR (Expected Shortfall).**

CVaR = the *average* loss on the days that breach the VaR threshold — it answers "how bad is bad?", which VaR alone doesn't.


In [ ]:
breach_returns = portfolio_returns[portfolio_returns <= historical_var]
cvar = breach_returns.mean()

print(f"Historical VaR ({confidence:.0%}):  {historical_var:.4%}")
print(f"CVaR / Expected Shortfall:     {cvar:.4%}")


**Beginner checkpoint:** VaR tells you a *threshold* loss you're unlikely to exceed at a given confidence — it says nothing about how severe losses beyond that threshold could be. CVaR fixes exactly that gap, which is why regulators (e.g., under Basel III/FRTB) have shifted toward requiring CVaR/Expected Shortfall alongside or instead of VaR.

---
## Wrap-up

You've now built, in order:
1. A data pipeline + technical indicator (SMA)
2. A backtested trading strategy with correct lookahead-bias handling
3. A derivatives pricing model with Greeks
4. A portfolio optimizer (mean-variance / efficient frontier)
5. Three different risk measures (VaR/CVaR)

**Natural next step:** combine Project 2's backtesting framework with Project 4's optimized weights to backtest a *multi-asset, risk-managed* strategy — which is close to what a real quant research pipeline looks like end to end.
